# Comparing W&B runs: parallel coordinates, correlation plots, and run diff

Once you have several training runs logged to a W&B project, the interesting questions become comparative: *which hyperparameters drove accuracy up?* and *where do my runs disagree?* This notebook walks through three comparison views using the Python SDK and the `wandb` API:

1. **Parallel coordinates** — see how hyperparameter ranges map to the output metric.
2. **Correlation plot** — quantify linear relationships between config, metrics, and runtime.
3. **Run diff** — programmatically diff two runs' configs and summary metrics.

The notebook assumes you have runs already logged to a project (e.g. from a sweep). It does not require a live UI session — everything is driven from the API.

## Setup

Import the SDK and a couple of helpers for the analysis. We pull runs with the `wandb.Api`.

In [ ]:
import wandb
import pandas as pd
import numpy as np
from wandb.apis import PublicApi

api = PublicApi()
PROJECT = "mlops-kit-demo"
ENTITY = wandb.Api().viewer.server_info.get("entity") or None
print(f"Connected to project: {PROJECT}")

## 1. Pull runs into a table

We query the last several runs and flatten each run's `config` and `summary` into columns so we can feed them to the comparison views.

In [ ]:
runs = api.runs(f"{ENTITY}/{PROJECT}", per_page=50)
rows = []
for r in runs:
    row = {}
    row["name"] = r.name
    row["state"] = r.state
    # config values are nested under .value in the W&B API
    for k, v in r.config.items():
        if isinstance(v, dict) and "value" in v:
            row[k] = v["value"]
        else:
            row[k] = v
    # summary metrics are flat
    for k, v in r.summary.items():
        if isinstance(v, (int, float)):
            row[k] = v
    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head()

## 2. Parallel coordinates

W&B renders a parallel-coordinates plot directly when you log a `wandb.Table` with the right columns. The trick is to set the table's `default_visualization` to `"parallel_coordinates"` and include your output metric as the last column. Here we rebuild that table from our dataframe.

Pick the columns you care about: numeric hyperparameters plus the metric you want to optimize.

In [ ]:
param_cols = [c for c in ["alpha", "l1_ratio", "learning_rate", "max_depth"] if c in df.columns]
metric_cols = [c for c in ["accuracy", "eval_rmse", "loss", "_step"] if c in df.columns]
if not metric_cols:
    metric_cols = [df.select_dtypes("number").columns[-1]]

pc_cols = param_cols + metric_cols
pc_data = df[pc_cols].dropna()
print("Parallel coordinates columns:", pc_cols)
pc_data.head()

In [ ]:
import wandb

table = wandb.Table(dataframe=pc_data)
table = wandb.Table(columns=list(pc_data.columns), data=pc_data.values.tolist())
table.set_property("default_visualization", "parallel_coordinates")

with wandb.init(project=PROJECT, name="run-comparison-parallel-coords", job_type="analysis") as run:
    run.log({"parallel_coordinates": table})
print("Logged parallel-coordinates table. Open the run to interact with it.")

Reading it: each horizontal line is one run, threaded through the parameter axes and ending at the metric axis. Lines that bunch at high metric values localize the good hyperparameter ranges — exactly the kind of signal a sweep is meant to surface.

## 3. Correlation plot

Parallel coordinates show direction; a correlation matrix quantifies it. We compute Pearson correlations across numeric columns and log the matrix as a W&B heatmap.

In [ ]:
numeric = df.select_dtypes(include=[np.number]).dropna(axis=1, how="any")
corr = numeric.corr(method="pearson")
corr.round(2)

In [ ]:
import wandb

labels = list(corr.columns)
heatmap = wandb.plots.HeatMap(
    x_labels=labels,
    y_labels=labels,
    matrix_values=corr.values.tolist(),
    show_text=True,
)

with wandb.init(project=PROJECT, name="run-comparison-correlation", job_type="analysis") as run:
    run.log({"correlation": heatmap})
print("Logged correlation heatmap.")

Look for strong correlations between a parameter and your metric. A near-1.0 or -1.0 cell (off the diagonal) tells you which knob actually mattered — useful when the parallel-coordinates plot is too busy to read at a glance.

## 4. Run diff (programmatic)

Sometimes you want the exact delta between two runs — e.g. "what changed between the staging model and the production model?" We compare configs and summary metrics key-by-key.

In [ ]:
def flatten_config(cfg):
    out = {}
    for k, v in cfg.items():
        out[k] = v["value"] if isinstance(v, dict) and "value" in v else v
    return out

def diff_runs(run_a, run_b):
    a_cfg, b_cfg = flatten_config(run_a.config), flatten_config(run_b.config)
    keys = sorted(set(a_cfg) | set(b_cfg))
    print(f"=== Config diff: {run_a.name} vs {run_b.name} ===")
    for k in keys:
        if a_cfg.get(k) != b_cfg.get(k):
            print(f"  {k}: {a_cfg.get(k)} -> {b_cfg.get(k)}")

    a_sum, b_sum = run_a.summary, run_b.summary
    s_keys = sorted(set(a_sum) & set(b_sum))
    print(f"\n=== Metric diff ===")
    for k in s_keys:
        if isinstance(a_sum[k], (int, float)) and isinstance(b_sum[k], (int, float)):
            delta = b_sum[k] - a_sum[k]
            if delta != 0:
                print(f"  {k}: {a_sum[k]:.4f} -> {b_sum[k]:.4f} (Δ {delta:+.4f})")

runs = list(api.runs(f"{ENTITY}/{PROJECT}", per_page=10))
if len(runs) >= 2:
    diff_runs(runs[0], runs[1])
else:
    print("Need at least two runs to diff.")

## What I covered

Three complementary ways to compare runs: parallel coordinates for direction, a correlation heatmap for量化 (quantification), and a programmatic diff for exact deltas. Together they answer "which hyperparameters mattered" and "what changed between two runs" without leaving the notebook.